## Imports

In [1]:
import os
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

from sklearn.model_selection import StratifiedShuffleSplit

import warnings
warnings.filterwarnings('ignore')

In [24]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
## Connection
connection = presto.connect(
        
        host='presto-gateway.processing.data.production.internal',
#     presto.processing.yoda.run
#     presto-gateway.processing.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [4]:
cwd = os.getcwd()
print(cwd)
local_datasource = '/Users/E2074/local-datasets/customer/ads-monetisation/amazon-mintv'
print(local_datasource)

/Users/E2074/analytics/customer/Ads-Monetisation/amazon-minitv
/Users/E2074/local-datasets/customer/ads-monetisation/amazon-mintv


## Datasets & Parameter

In [5]:
## Parameter 
current_date = datetime.now()
start_date = '20240524'
end_date = '20240603'

# Convert date strings to datetime objects
start_dt = datetime.strptime(start_date, '%Y%m%d')
end_dt = datetime.strptime(end_date, '%Y%m%d')
segment_date = end_dt.strftime('%Y-%m-%d')
print(start_date, ' to ' ,end_date)

20240524  to  20240603


### Ads

In [6]:
# Initialize an empty DataFrame to hold the results
df_data = pd.DataFrame()

# Loop through each date in the range
current_dt = start_dt
while current_dt <= end_dt:
    # Format the current date as a string in 'YYYYMMDD' format
    current_date_str = current_dt.strftime('%Y%m%d')
    
    data_query = f"""
    
        WITH ads_base as (
    
        SELECT 
            distinct
            yyyymmdd,
            city,
            CASE 
            WHEN service = 'Bike' THEN 'Bike Lite' 
            WHEN service = 'Cab' THEN 'CabEconomy'
            ELSE service END service,
            userid,
            hh hour,
            pagename,
            CASE 
            WHEN pagename IN ('HomeScreen', 'CaptainSearchScreen') THEN pagename
            WHEN lower(slotname) LIKE '%ontheway%' THEN 'onTheWay'
            WHEN lower(slotname) LIKE '%arrived%' THEN 'arrived'
            WHEN lower(slotname) LIKE '%started%' THEN 'started'
            WHEN ltrim(orderstatus) IS NOT NULL THEN orderstatus
            END screen_slot_name,
            CASE WHEN eventName = 'Ad_Viewable_Impression' THEN adid END viewed,
            CASE WHEN eventName = 'Ad_Click' THEN adid END clicked
        FROM    
            canonical.iceberg_log_telemetry_ads_impressions_immutable_full
        
        WHERE  
            yyyymmdd = '{current_date_str}'
            AND eventname in ('Ad_Viewable_Impression', 'Ad_Click')
            AND responsetype = 'GAMBanner'
            AND city in ('Bangalore', 'Chennai', 'Delhi', 'Hyderabad', 'Mumbai')
            AND format in ('nativeVideoBanner')
            AND (service in ('Auto', 'Auto NCR', 'Bike', 'Bike Lite', 'Cab', 'CabEconomy', 'Link') or pagename = 'HomeScreen')
        ),
        
        segment AS (
        
        SELECT
            customer_id,
            taxi_ltr_segment ltr_segment,
            taxi_retention_segments retention_segment,
            taxi_lifetime_stage lifetime_stage,
            taxi_service_affinity service_affinity,
            gender_tag gender
        FROM
            datasets.iallocator_customer_segments
        WHERE 
            run_date = '{segment_date}'
        )
        
        SELECT 
            ads_base.*,
            coalesce(ads_base.service, 'HomeScreen') service_name,
            coalesce( ltr_segment, 'UNKNOWN')ltr_segment,
            coalesce( retention_segment, 'UNKNOWN') retention_segment,
            coalesce( lifetime_stage, 'UNKNOWN') lifetime_stage,
            coalesce( service_affinity, 'UNKNOWN') service_affinity,
            coalesce( gender, 'UNKNOWN')gender
        FROM 
            ads_base
        LEFT JOIN 
            segment
            ON ads_base.userid = segment.customer_id
    """

    # Execute the query and get the result as a DataFrame
    df_day = pd.read_sql(data_query, connection)
    
    # Concatenate the result with the existing DataFrame
    df_data = pd.concat([df_data, df_day], ignore_index=True)
    
    print(current_date_str)
    # Move to the next date
    current_dt += timedelta(days=1)


df_data.to_csv( local_datasource + '/analysis_data_{}_{}.csv'.format(start_date, end_date), index=False)

20240524
20240525
20240526
20240527
20240528
20240529
20240530
20240531
20240601
20240602
20240603


In [6]:
df_data = pd.read_csv( local_datasource + '/analysis_data_{}_{}.csv'.format(start_date, end_date))

### Appography

In [11]:
appography_query = f"""

    WITH ads_base as (
        
        SELECT 
            DISTINCT userid
        FROM    
            canonical.iceberg_log_telemetry_ads_impressions_immutable_full
        
        WHERE  
            yyyymmdd >= '{start_date}'
            AND yyyymmdd <= '{end_date}'
            AND eventname in ('Ad_Viewable_Impression', 'Ad_Click')
            AND responsetype = 'GAMBanner'
            AND city in ('Bangalore', 'Chennai', 'Delhi', 'Hyderabad', 'Mumbai')
            AND format in ('nativeVideoBanner')
            AND (service in ('Auto', 'Auto NCR', 'Bike', 'Bike Lite', 'Cab', 'CabEconomy', 'Link') or pagename = 'HomeScreen')
        ),
        
        appography_data AS (
        SELECT
            user_id,
            app_list
        FROM 
            (
            SELECT
                yyyymmdd,
                id user_id,
                appography_data as app_list,
                row_number() over(partition by id order by updated_epoch desc) updated_seq
            FROM
                customer.info_customer_installed_apps_immutable
            WHERE
                yyyymmdd >= date_format(current_date - INTERVAL '50' DAY, '%Y%m%d')
                and yyyymmdd <= date_format(current_date, '%Y%m%d')
                and id IN (SELECT userid FROM ads_base)
            ) 
        WHERE
            updated_seq = 1
        )
        
        SELECT
            *
        FROM 
            appography_data
"""

df_appography = pd.read_sql(appography_query, connection)
df_appography.to_csv( local_datasource + '/appography_data_{}_{}.csv'.format(start_date, end_date), index=False)

In [7]:
df_appography = pd.read_csv( local_datasource + '/appography_data_{}_{}.csv'.format(start_date, end_date))

### Category mapping

In [8]:
# df_category = pd.read_clipboard()

In [9]:
# df_category.to_csv( local_datasource + '/category_mapping.csv', index=False)
df_category = pd.read_csv( local_datasource + '/category_mapping.csv')
df_category.head(3)

,category,app_name
0,Dating,Aisle
1,Dating,Bumble
2,Dating,Coffee Meets Bagel


## User defined function

In [10]:
agg_dict = {
    'customers': ('userid', 'nunique'),
    'viewed': ('viewed', 'nunique'),
    'clicked': ('clicked', 'nunique'),
}

def calculate_percentage(df, numerator, denominator):
    
    percentage = (df[numerator] * 100.00 / df[denominator]).round(2)
    
    return percentage

def calculate_percentage_columns(df):
        
    df['cta'] = calculate_percentage(df, 'clicked', 'viewed')
    
    return df

##Define a function to extract values between double quotations and trim extra spaces
def extract_and_trim(text):
    
    return [val.strip() for val in re.findall(r'"(.*?)"', text)]

## Analysis

In [9]:
df_data.head(5)

,yyyymmdd,city,service,userid,hour,pagename,screen_slot_name,viewed,clicked,service_name,ltr_segment,retention_segment,lifetime_stage,service_affinity,gender
0,20240524,Hyderabad,,6275e949dce696199d55340b,22,HomeScreen,HomeScreen,ad_6275e949dce696199d55340b_1716568565212,None,,PHH,ELITE,COMMITTED,NO_AFFINITY,MALE
1,20240524,Bangalore,CabEconomy,5de287bcd472d62e9459774f,16,CaptainSearchScreen,CaptainSearchScreen,ad_5de287bcd472d62e9459774f_1716524715744,None,CabEconomy,PHH,SILVER,CHURN_OTB,ONLY_CAB,FEMALE
2,20240524,Bangalore,Auto,6645cf59a31b59235065f83a,23,CaptainSearchScreen,CaptainSearchScreen,ad_6645cf59a31b59235065f83a_1716573150861,None,Auto,HH,HH,HANDHOLDING,LINK_AUTO,MALE
3,20240524,Mumbai,,621c55ba61482dc318a02a18,23,HomeScreen,HomeScreen,ad_621c55ba61482dc318a02a18_1716574639728,None,,PHH,INACTIVE,INACTIVE,UNKNOWN,MALE
4,20240524,Bangalore,Auto,65ec2eae283274d33fb1f4a9,08,CaptainSearchScreen,CaptainSearchScreen,ad_65ec2eae283274d33fb1f4a9_1716440888836,None,Auto,PHH,PLATINUM,HOOK,ONLY_AUTO,FEMALE


In [10]:
df_data.columns

Index(['yyyymmdd', 'city', 'service', 'userid', 'hour', 'pagename',
       'screen_slot_name', 'viewed', 'clicked', 'service_name', 'ltr_segment',
       'retention_segment', 'lifetime_stage', 'service_affinity', 'gender'],
      dtype='object')

In [11]:
df_data['service'].fillna(df_data['pagename'], inplace=True)
df_data

,yyyymmdd,city,service,userid,hour,pagename,screen_slot_name,viewed,clicked,service_name,ltr_segment,retention_segment,lifetime_stage,service_affinity,gender
0,20240524,Hyderabad,,6275e949dce696199d55340b,22,HomeScreen,HomeScreen,ad_6275e949dce696199d55340b_1716568565212,None,,PHH,ELITE,COMMITTED,NO_AFFINITY,MALE
1,20240524,Bangalore,CabEconomy,5de287bcd472d62e9459774f,16,CaptainSearchScreen,CaptainSearchScreen,ad_5de287bcd472d62e9459774f_1716524715744,None,CabEconomy,PHH,SILVER,CHURN_OTB,ONLY_CAB,FEMALE
2,20240524,Bangalore,Auto,6645cf59a31b59235065f83a,23,CaptainSearchScreen,CaptainSearchScreen,ad_6645cf59a31b59235065f83a_1716573150861,None,Auto,HH,HH,HANDHOLDING,LINK_AUTO,MALE
3,20240524,Mumbai,,621c55ba61482dc318a02a18,23,HomeScreen,HomeScreen,ad_621c55ba61482dc318a02a18_1716574639728,None,,PHH,INACTIVE,INACTIVE,UNKNOWN,MALE
4,20240524,Bangalore,Auto,65ec2eae283274d33fb1f4a9,08,CaptainSearchScreen,CaptainSearchScreen,ad_65ec2eae283274d33fb1f4a9_1716440888836,None,Auto,PHH,PLATINUM,HOOK,ONLY_AUTO,FEMALE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20187204,20240603,Hyderabad,Link,63f0f9a0dfeed30718e3e071,10,PostOrderScreen,onTheWay,ad_63f0f9a0dfeed30718e3e071_1717391914381,None,Link,PHH,SILVER,HOOK,LINK_AUTO,MALE
20187205,20240603,Bangalore,CabEconomy,6651df52b46daa1619cfddee,18,PostOrderScreen,arrived,ad_6651df52b46daa1619cfddee_1717421206890,None,CabEconomy,PHH,PLATINUM,COMMITTED,ONLY_AUTO,MALE
20187206,20240603,Chennai,Auto,64b8adf3a6ff71c1b1dbe1f7,09,PostOrderScreen,started,ad_64b8adf3a6ff71c1b1dbe1f7_1717387682347,None,Auto,PHH,PLATINUM,SUSTENANCE,NO_AFFINITY,MALE
20187207,20240603,Bangalore,Link,5d6a29d1a24c96105e2ff242,09,CaptainSearchScreen,CaptainSearchScreen,ad_5d6a29d1a24c96105e2ff242_1717387233811,None,Link,PHH,SILVER,HOOK,ONLY_AUTO,MALE


### Pivot view

In [12]:
df_analysis1 = df_data\
.groupby(['yyyymmdd'])\
.agg(**agg_dict)\
.reset_index()

df_analysis1 = calculate_percentage_columns(df_analysis1)
df_analysis1

,yyyymmdd,customers,viewed,clicked,cta
0,20240524,538400,998921,4607,0.46
1,20240525,625422,1364432,4765,0.35
2,20240526,594464,1326198,5844,0.44
3,20240527,706307,1746602,7641,0.44
4,20240528,657464,1865709,4314,0.23
5,20240529,742847,1981602,5704,0.29
6,20240530,804400,2545637,9066,0.36
7,20240531,849958,2563930,8837,0.34
8,20240601,769214,2226874,12039,0.54
9,20240602,552875,1191621,5852,0.49


In [13]:
df_analysis2 = df_data\
.groupby(['city'])\
.agg(**agg_dict)\
.reset_index()

df_analysis2 = calculate_percentage_columns(df_analysis2)
df_analysis2

,city,customers,viewed,clicked,cta
0,Bangalore,951598,4884758,25993,0.53
1,Chennai,567673,1998267,6931,0.35
2,Delhi,856151,3520347,11367,0.32
3,Hyderabad,1379540,7097633,25270,0.36
4,Mumbai,182212,636608,2274,0.36


In [14]:
df_analysis3 = df_data\
.groupby(['service_name'])\
.agg(**agg_dict)\
.reset_index()

df_analysis3 = calculate_percentage_columns(df_analysis3)
df_analysis3

,service_name,customers,viewed,clicked,cta
0,,1785458,2396854,3000,0.13
1,Auto,1872890,7434396,35185,0.47
2,Auto NCR,7024,18722,87,0.46
3,Bike Lite,318896,981764,3700,0.38
4,CabEconomy,582579,1686543,9284,0.55
5,Link,1454130,5877908,20757,0.35


In [15]:
df_analysis4 = df_data\
.groupby(['pagename'])\
.agg(**agg_dict)\
.reset_index()

df_analysis4 = calculate_percentage_columns(df_analysis4)
df_analysis4

,pagename,customers,viewed,clicked,cta
0,CaptainSearchScreen,2112268,3330564,35751,1.07
1,HomeScreen,1785458,2396854,3000,0.13
2,PostOrderScreen,2937543,12513889,33081,0.26


In [16]:
df_analysis5 = df_data\
.groupby(['pagename','screen_slot_name'])\
.agg(**agg_dict)\
.reset_index()

df_analysis5 = calculate_percentage_columns(df_analysis5)
df_analysis5

,pagename,screen_slot_name,customers,viewed,clicked,cta
0,CaptainSearchScreen,CaptainSearchScreen,2112268,3330564,35751,1.07
1,HomeScreen,HomeScreen,1785458,2396854,3000,0.13
2,PostOrderScreen,arrived,1479571,2715573,2630,0.10
3,PostOrderScreen,onTheWay,2392191,6151886,16070,0.26
4,PostOrderScreen,started,1760385,3646807,14381,0.39


In [17]:
df_analysis_6 = df_data\
.groupby(['ltr_segment'])\
.agg(**agg_dict)\
.reset_index()

df_analysis_6 = calculate_percentage_columns(df_analysis_6)
df_analysis_6

,ltr_segment,customers,viewed,clicked,cta
0,HH,736221,2273007,10234,0.45
1,PHH,2992567,15575957,59856,0.38
2,UNKNOWN,173054,279193,1741,0.62


In [18]:
df_analysis_7 = df_data\
.groupby(['retention_segment'])\
.agg(**agg_dict)\
.reset_index()

df_analysis_7 = calculate_percentage_columns(df_analysis_7)
df_analysis_7

,retention_segment,customers,viewed,clicked,cta
0,DORMANT,314642,490970,3077,0.63
1,ELITE,412226,4184774,15183,0.36
2,GOLD,872379,4029915,15476,0.38
3,HH,572489,2025462,8724,0.43
4,INACTIVE,160775,241744,1478,0.61
5,PLATINUM,720497,4353573,16457,0.38
6,PRIME,28012,342115,1176,0.34
7,SILVER,647768,2180418,8519,0.39
8,UNKNOWN,173054,279193,1741,0.62


In [19]:
df_analysis_8 = df_data\
.groupby(['lifetime_stage'])\
.agg(**agg_dict)\
.reset_index()

df_analysis_8 = calculate_percentage_columns(df_analysis_8)
df_analysis_8

,lifetime_stage,customers,viewed,clicked,cta
0,CHURN_OTB,145615,432384,1464,0.34
1,COMMITTED,897106,7454210,27602,0.37
2,DETOX,11531,74081,244,0.33
3,DORMANT,314642,490970,3077,0.63
4,HANDHOLDING,572489,2025462,8724,0.43
5,HOOK,1096610,4916099,18407,0.37
6,INACTIVE,160775,241744,1478,0.61
7,SOFT_CHURN,51123,120804,486,0.40
8,SUSTENANCE,478897,2093217,8608,0.41
9,UNKNOWN,173054,279193,1741,0.62


In [20]:
df_analysis_9 = df_data\
.groupby(['service_affinity'])\
.agg(**agg_dict)\
.reset_index()

df_analysis_9 = calculate_percentage_columns(df_analysis_9)
df_analysis_9

,service_affinity,customers,viewed,clicked,cta
0,AUTO_CAB,109932,494128,2276,0.46
1,LINK_AUTO,346609,1706095,6390,0.37
2,LINK_CAB,39783,150078,599,0.40
3,NO_AFFINITY,349192,1752417,7165,0.41
4,ONLY_AUTO,1371517,6607872,28812,0.44
5,ONLY_CAB,193136,784524,3300,0.42
6,ONLY_LINK,1247783,6267605,21265,0.34
7,UNKNOWN,243890,365445,2024,0.55


In [21]:
df_analysis_10 = df_data\
.groupby(['gender'])\
.agg(**agg_dict)\
.reset_index()

df_analysis_10 = calculate_percentage_columns(df_analysis_10)
df_analysis_10

,gender,customers,viewed,clicked,cta
0,FEMALE,1076431,5304946,24025,0.45
1,MALE,2630784,12195363,45175,0.37
2,OTHERS,12874,67509,395,0.59
3,UNKNOWN,181753,560341,2236,0.40


In [22]:
df_analysis_11 = df_data\
.groupby(['city', 'service_name','ltr_segment'])\
.agg(**agg_dict)\
.reset_index()

df_analysis_11 = calculate_percentage_columns(df_analysis_11)
df_analysis_11

,city,service_name,ltr_segment,customers,viewed,clicked,cta
0,Bangalore,,HH,66402,78650,108,0.14
1,Bangalore,,PHH,303468,421744,369,0.09
2,Bangalore,,UNKNOWN,20546,22938,55,0.24
3,Bangalore,Auto,HH,85472,258472,1953,0.76
4,Bangalore,Auto,PHH,486860,2197147,13738,0.63
5,Bangalore,Auto,UNKNOWN,11566,18187,332,1.83
6,Bangalore,Bike Lite,HH,5247,8028,58,0.72
7,Bangalore,Bike Lite,PHH,30367,55445,358,0.65
8,Bangalore,Bike Lite,UNKNOWN,460,574,6,1.05
9,Bangalore,CabEconomy,HH,24603,65028,491,0.76


In [26]:

pd.pivot_table(df_analysis_11, values= ['cta'] , index=['city', 'ltr_segment'],
                       columns=['service_name'])

cta                                          
service_name                 Auto Auto NCR Bike Lite CabEconomy  Link
city      ltr_segment                                                
Bangalore HH           0.14  0.76      NaN      0.72       0.76  0.49
          PHH          0.09  0.63      NaN      0.65       0.70  0.38
          UNKNOWN      0.24  1.83      NaN      1.05       1.78  0.99
Chennai   HH           0.23  0.41      NaN      0.58       0.52  0.42
          PHH          0.14  0.36      NaN      0.55       0.38  0.33
          UNKNOWN      0.28  1.02      NaN      1.82       1.13  0.81
Delhi     HH           0.16  0.46     0.68      0.50       0.49  0.41
          PHH          0.10  0.32     0.41      0.34       0.39  0.33
          UNKNOWN      0.27  0.76     0.35      1.26       0.64  0.78
Hyderabad HH           0.18  0.45      NaN      0.45       0.57  0.41
          PHH          0.11  0.39      NaN      0.31       0.52  0.32
          UNKNOWN      0.21  0.95      NaN      0.82       1.49  0.87
Mumbai    HH           0.17  0.49      NaN       NaN       0.64  0.00
          PHH          0.08  0.36      NaN      0.00       0.62  1.53
          UNKNOWN      0.23  0.87      NaN       NaN       1.05  0.00

In [23]:
df_analysis12 = df_data\
.groupby(['city', 'service_name', 'pagename','screen_slot_name'])\
.agg(**agg_dict)\
.reset_index()

df_analysis12 = calculate_percentage_columns(df_analysis12)
df_analysis12

,city,service_name,pagename,screen_slot_name,customers,viewed,clicked,cta
0,Bangalore,,HomeScreen,HomeScreen,390416,523332,532,0.10
1,Bangalore,Auto,CaptainSearchScreen,CaptainSearchScreen,454903,684673,11737,1.71
2,Bangalore,Auto,PostOrderScreen,arrived,217805,380077,306,0.08
3,Bangalore,Auto,PostOrderScreen,onTheWay,367981,839874,1935,0.23
4,Bangalore,Auto,PostOrderScreen,started,284424,588471,2045,0.35
5,Bangalore,Bike Lite,CaptainSearchScreen,CaptainSearchScreen,34157,44657,356,0.80
6,Bangalore,Bike Lite,PostOrderScreen,arrived,2559,3814,3,0.08
7,Bangalore,Bike Lite,PostOrderScreen,onTheWay,5417,10619,37,0.35
8,Bangalore,Bike Lite,PostOrderScreen,started,3045,5084,26,0.51
9,Bangalore,CabEconomy,CaptainSearchScreen,CaptainSearchScreen,114498,131530,2201,1.67


In [24]:

pd.pivot_table(df_analysis12, values= ['cta'] , index=['city', 'service_name'],
                       columns=['screen_slot_name'])

cta                                    
screen_slot_name       CaptainSearchScreen HomeScreen arrived onTheWay started
city      service_name                                                        
Bangalore                              NaN       0.10     NaN      NaN     NaN
          Auto                        1.71        NaN    0.08     0.23    0.35
          Bike Lite                   0.80        NaN    0.08     0.35    0.51
          CabEconomy                  1.67        NaN    0.14     0.37    0.35
          Link                        0.69        NaN    0.08     0.28    0.43
Chennai                                NaN       0.16     NaN      NaN     NaN
          Auto                        0.82        NaN    0.07     0.11    0.39
          Bike Lite                   0.79        NaN    0.07     0.18    0.42
          CabEconomy                  0.91        NaN    0.10     0.19    0.31
          Link                        0.76        NaN    0.04     0.15    0.37
Delhi                                  NaN       0.13     NaN      NaN     NaN
          Auto                        0.78        NaN    0.12     0.33    0.40
          Auto NCR                    1.12        NaN    0.15     0.49    0.41
          Bike Lite                   0.88        NaN    0.12     0.29    0.51
          CabEconomy                  0.94        NaN    0.17     0.43    0.40
          Link                        0.89        NaN    0.13     0.31    0.49
Hyderabad                              NaN       0.12     NaN      NaN     NaN
          Auto                        0.89        NaN    0.09     0.27    0.40
          Bike Lite                   0.52        NaN    0.10     0.19    0.36
          CabEconomy                  1.23        NaN    0.15     0.32    0.30
          Link                        0.65        NaN    0.08     0.22    0.38
Mumbai                                 NaN       0.13     NaN      NaN     NaN
          Auto                        0.79        NaN    0.08     0.19    0.39
          Bike Lite                   0.00        NaN    0.00     0.00    0.00
          CabEconomy                  1.31        NaN    0.18     0.32    0.43
          Link                         NaN        NaN    0.00     1.12    1.75

In [28]:
df_analysis_13 = df_data\
.groupby(['city', 'service_name','pagename'])\
.agg(**agg_dict)\
.reset_index()

df_analysis_13 = calculate_percentage_columns(df_analysis_13)
df_analysis_13

,city,service_name,pagename,customers,viewed,clicked,cta
0,Bangalore,,HomeScreen,390416,523332,532,0.10
1,Bangalore,Auto,CaptainSearchScreen,454903,684673,11737,1.71
2,Bangalore,Auto,PostOrderScreen,452580,1808379,4286,0.24
3,Bangalore,Bike Lite,CaptainSearchScreen,34157,44657,356,0.80
4,Bangalore,Bike Lite,PostOrderScreen,6451,19516,66,0.34
5,Bangalore,CabEconomy,CaptainSearchScreen,114498,131530,2201,1.67
6,Bangalore,CabEconomy,PostOrderScreen,110262,312713,994,0.32
7,Bangalore,Link,CaptainSearchScreen,247796,414158,2850,0.69
8,Bangalore,Link,PostOrderScreen,296334,1098238,3102,0.28
9,Chennai,,HomeScreen,194928,240290,389,0.16


In [31]:
pd.pivot_table(df_analysis_13, values= ['cta'] , index=['city', 'pagename'],
                       columns=['service_name'])

cta                                          
service_name                         Auto Auto NCR Bike Lite CabEconomy  Link
city      pagename                                                           
Bangalore CaptainSearchScreen   NaN  1.71      NaN      0.80       1.67  0.69
          HomeScreen           0.10   NaN      NaN       NaN        NaN   NaN
          PostOrderScreen       NaN  0.24      NaN      0.34       0.32  0.28
Chennai   CaptainSearchScreen   NaN  0.82      NaN      0.79       0.91  0.76
          HomeScreen           0.16   NaN      NaN       NaN        NaN   NaN
          PostOrderScreen       NaN  0.23      NaN      0.25       0.21  0.19
Delhi     CaptainSearchScreen   NaN  0.78     1.12      0.88       0.94  0.89
          HomeScreen           0.13   NaN      NaN       NaN        NaN   NaN
          PostOrderScreen       NaN  0.30     0.40      0.30       0.37  0.31
Hyderabad CaptainSearchScreen   NaN  0.89      NaN      0.52       1.23  0.65
          HomeScreen           0.12   NaN      NaN       NaN        NaN   NaN
          PostOrderScreen       NaN  0.27      NaN      0.21       0.27  0.23
Mumbai    CaptainSearchScreen   NaN  0.79      NaN      0.00       1.31   NaN
          HomeScreen           0.13   NaN      NaN       NaN        NaN   NaN
          PostOrderScreen       NaN  0.23      NaN      0.00       0.34  1.12

In [32]:
df_analysis_14 = df_data\
.groupby(['city', 'service_name','gender'])\
.agg(**agg_dict)\
.reset_index()

df_analysis_14 = calculate_percentage_columns(df_analysis_14)
df_analysis_14

,city,service_name,gender,customers,viewed,clicked,cta
0,Bangalore,,FEMALE,110548,152290,129,0.08
1,Bangalore,,MALE,256803,342942,349,0.10
2,Bangalore,,OTHERS,1322,1822,3,0.16
3,Bangalore,,UNKNOWN,21743,26279,51,0.19
4,Bangalore,Auto,FEMALE,217791,995033,7289,0.73
5,Bangalore,Auto,MALE,346665,1397347,8120,0.58
6,Bangalore,Auto,OTHERS,2006,9670,89,0.92
7,Bangalore,Auto,UNKNOWN,17436,71757,525,0.73
8,Bangalore,Bike Lite,FEMALE,6282,12132,106,0.87
9,Bangalore,Bike Lite,MALE,28649,49914,299,0.60


In [33]:
pd.pivot_table(df_analysis_14, values= ['cta'] , index=['city', 'gender'],
                       columns=['service_name'])

cta                                          
service_name             Auto Auto NCR Bike Lite CabEconomy  Link
city      gender                                                 
Bangalore FEMALE   0.08  0.73      NaN      0.87       0.88  0.43
          MALE     0.10  0.58      NaN      0.60       0.65  0.39
          OTHERS   0.16  0.92      NaN      1.28       0.65  0.68
          UNKNOWN  0.19  0.73      NaN      0.79       0.85  0.44
Chennai   FEMALE   0.12  0.39      NaN      0.52       0.47  0.41
          MALE     0.18  0.36      NaN      0.57       0.39  0.33
          OTHERS   0.28  0.44      NaN      0.79       1.02  0.42
          UNKNOWN  0.24  0.32      NaN      0.66       0.31  0.34
Delhi     FEMALE   0.10  0.37     0.64      0.38       0.45  0.38
          MALE     0.12  0.33     0.38      0.34       0.40  0.35
          OTHERS   0.19  0.61     0.00      0.71       0.51  0.51
          UNKNOWN  0.27  0.34     0.44      0.45       0.41  0.40
Hyderabad FEMALE   0.11  0.43      NaN      0.36       0.58  0.35
          MALE     0.13  0.38      NaN      0.32       0.51  0.32
          OTHERS   0.18  0.59      NaN      0.86       0.87  0.38
          UNKNOWN  0.15  0.41      NaN      0.31       0.52  0.34
Mumbai    FEMALE   0.11  0.42      NaN      0.00       0.74  6.67
          MALE     0.12  0.37      NaN      0.00       0.62  0.00
          OTHERS   0.00  1.02      NaN       NaN       1.69  0.00
          UNKNOWN  0.20  0.44      NaN      0.00       0.48  0.00

In [34]:
df_analysis_15 = df_data\
.groupby(['city', 'retention_segment'])\
.agg(**agg_dict)\
.reset_index()

df_analysis_15 = calculate_percentage_columns(df_analysis_15)
df_analysis_15

,city,retention_segment,customers,viewed,clicked,cta
0,Bangalore,DORMANT,82103,131119,1381,1.05
1,Bangalore,ELITE,101593,1183171,5597,0.47
2,Bangalore,GOLD,217404,1085186,5589,0.52
3,Bangalore,HH,131183,506715,2805,0.55
4,Bangalore,INACTIVE,37102,56622,605,1.07
5,Bangalore,PLATINUM,182394,1216698,6073,0.50
6,Bangalore,PRIME,6705,85190,382,0.45
7,Bangalore,SILVER,157961,562104,2959,0.53
8,Bangalore,UNKNOWN,35153,57959,602,1.04
9,Chennai,DORMANT,27556,37529,233,0.62


In [35]:
pd.pivot_table(df_analysis_15, values= ['cta'] , index=['city'],
                       columns=['retention_segment'])

cta                                                   \
retention_segment DORMANT ELITE  GOLD    HH INACTIVE PLATINUM PRIME SILVER   
city                                                                         
Bangalore            1.05  0.47  0.52  0.55     1.07     0.50  0.45   0.53   
Chennai              0.62  0.34  0.33  0.38     0.61     0.32  0.32   0.33   
Delhi                0.33  0.28  0.31  0.39     0.35     0.30  0.31   0.34   
Hyderabad            0.54  0.33  0.35  0.39     0.56     0.34  0.30   0.35   
Mumbai               0.40  0.30  0.36  0.40     0.41     0.30  0.34   0.31   

                           
retention_segment UNKNOWN  
city                       
Bangalore            1.04  
Chennai              0.65  
Delhi                0.46  
Hyderabad            0.62  
Mumbai               0.43

## Appography category vs Screen

In [11]:
df_data.head(3)

,yyyymmdd,city,service,userid,hour,pagename,screen_slot_name,viewed,clicked,service_name,ltr_segment,retention_segment,lifetime_stage,service_affinity,gender
0,20240524,Hyderabad,NaN,6275e949dce696199d55340b,22,HomeScreen,HomeScreen,ad_6275e949dce696199d55340b_1716568565212,NaN,NaN,PHH,ELITE,COMMITTED,NO_AFFINITY,MALE
1,20240524,Bangalore,CabEconomy,5de287bcd472d62e9459774f,16,CaptainSearchScreen,CaptainSearchScreen,ad_5de287bcd472d62e9459774f_1716524715744,NaN,CabEconomy,PHH,SILVER,CHURN_OTB,ONLY_CAB,FEMALE
2,20240524,Bangalore,Auto,6645cf59a31b59235065f83a,23,CaptainSearchScreen,CaptainSearchScreen,ad_6645cf59a31b59235065f83a_1716573150861,NaN,Auto,HH,HH,HANDHOLDING,LINK_AUTO,MALE


In [12]:
df_data.columns

Index(['yyyymmdd', 'city', 'service', 'userid', 'hour', 'pagename',
       'screen_slot_name', 'viewed', 'clicked', 'service_name', 'ltr_segment',
       'retention_segment', 'lifetime_stage', 'service_affinity', 'gender'],
      dtype='object')

In [17]:
df_data.shape, df_data.userid.nunique()

((20187209, 15), 3901841)

In [14]:
df_appography.head(1)

,user_id,app_list
0,5d0096dbc0b26018f5e280e5,"[""Adobe Lightroom"","" Airtel Thanks"","" Amazon A..."


### Appography work-around

In [15]:
df_category.head(1)

,category,app_name
0,Dating,Aisle


In [16]:
df_category['category'] = df_category['category'].str.lower()
df_category['app_name'] = df_category['app_name'].str.lower()
df_category.head(1)

,category,app_name
0,dating,aisle


In [20]:
df_customer_installed_apps = df_appography[['user_id', 'app_list']].copy()
df_customer_installed_apps['app_list'] = df_customer_installed_apps['app_list'].apply(lambda x: extract_and_trim(x))

In [21]:
df_customer_installed_apps.head(1)

,user_id,app_list
0,5d0096dbc0b26018f5e280e5,"[Adobe Lightroom, Airtel Thanks, Amazon Alexa,..."


In [22]:
## Explode list and map all the categories
df_customer_apps_explode = df_customer_installed_apps.explode('app_list')
df_customer_apps_explode['app_list'] = df_customer_apps_explode['app_list'].str.lower()

In [25]:
df_customer_apps_explode\
    .groupby(['app_list'])\
    .agg(customers = ('user_id','nunique'))\
    .sort_values(['customers'],ascending=False)\
    .reset_index()

,app_list,customers
0,google maps,3872227
1,youtube,3865715
2,gmail,3864923
3,whatsapp,3778263
4,google photos,3695566
5,instagram,3126859
6,facebook,2671660
7,truecaller,2591337
8,google calendar,2448593
9,flipkart,2437591


In [26]:
df_customer_apps_mapped = pd.merge(df_customer_apps_explode, 
                                   df_category, 
                                   how = 'left', 
                                   left_on = 'app_list', 
                                   right_on = 'app_name')

df_customer_apps_mapped.head(5)

,user_id,app_list,category,app_name
0,5d0096dbc0b26018f5e280e5,adobe lightroom,tools,adobe lightroom
1,5d0096dbc0b26018f5e280e5,airtel thanks,finance_transactions,airtel thanks
2,5d0096dbc0b26018f5e280e5,amazon alexa,tools,amazon alexa
3,5d0096dbc0b26018f5e280e5,amazon prime video,ott,amazon prime video
4,5d0096dbc0b26018f5e280e5,apple music,streaming_music,apple music


In [40]:
df_customer_apps_mapped.shape

(119113853, 4)

In [41]:
df_customer_category = df_customer_apps_mapped[['user_id', 'category']].drop_duplicates()
df_customer_category.shape

(39678964, 2)

### Appography & Ads 

In [43]:
df_customer_category.head(2)

,user_id,category
0,5d0096dbc0b26018f5e280e5,tools
1,5d0096dbc0b26018f5e280e5,finance_transactions


In [45]:
df_data.head(2)

,yyyymmdd,city,service,userid,hour,pagename,screen_slot_name,viewed,clicked,service_name,ltr_segment,retention_segment,lifetime_stage,service_affinity,gender
0,20240524,Hyderabad,NaN,6275e949dce696199d55340b,22,HomeScreen,HomeScreen,ad_6275e949dce696199d55340b_1716568565212,NaN,NaN,PHH,ELITE,COMMITTED,NO_AFFINITY,MALE
1,20240524,Bangalore,CabEconomy,5de287bcd472d62e9459774f,16,CaptainSearchScreen,CaptainSearchScreen,ad_5de287bcd472d62e9459774f_1716524715744,NaN,CabEconomy,PHH,SILVER,CHURN_OTB,ONLY_CAB,FEMALE


In [46]:
df_merge = pd.merge(df_data[['yyyymmdd', 'screen_slot_name', 'userid', 'viewed', 'clicked']], 
                    df_customer_category,
                    how='left',
                    left_on='userid',
                    right_on='user_id'
                   )

In [47]:
df_merge.head()

,yyyymmdd,screen_slot_name,userid,viewed,clicked,user_id,category
0,20240524,HomeScreen,6275e949dce696199d55340b,ad_6275e949dce696199d55340b_1716568565212,NaN,6275e949dce696199d55340b,delivery_grocery
1,20240524,HomeScreen,6275e949dce696199d55340b,ad_6275e949dce696199d55340b_1716568565212,NaN,6275e949dce696199d55340b,finance_transactions
2,20240524,HomeScreen,6275e949dce696199d55340b,ad_6275e949dce696199d55340b_1716568565212,NaN,6275e949dce696199d55340b,social
3,20240524,HomeScreen,6275e949dce696199d55340b,ad_6275e949dce696199d55340b_1716568565212,NaN,6275e949dce696199d55340b,tools
4,20240524,HomeScreen,6275e949dce696199d55340b,ad_6275e949dce696199d55340b_1716568565212,NaN,6275e949dce696199d55340b,ecommerce


In [49]:
df_merge.shape

(212722804, 7)

In [50]:
df_analysis1a = df_merge\
.groupby(['category', 'screen_slot_name'])\
.agg(**agg_dict)\
.reset_index()

df_analysis1a = calculate_percentage_columns(df_analysis1a)
df_analysis1a

,category,screen_slot_name,customers,viewed,clicked,cta
0,dating,CaptainSearchScreen,84613,151863,1497,0.99
1,dating,HomeScreen,69411,100122,91,0.09
2,dating,arrived,57454,110794,97,0.09
3,dating,onTheWay,96898,282207,569,0.20
4,dating,started,66174,142094,391,0.28
5,delivery_food,CaptainSearchScreen,1452183,2356580,24563,1.04
6,delivery_food,HomeScreen,1153403,1574366,1626,0.10
7,delivery_food,arrived,973021,1801611,1589,0.09
8,delivery_food,onTheWay,1589578,4171552,10014,0.24
9,delivery_food,started,1167119,2452955,8492,0.35


In [52]:
pd.pivot_table(df_analysis1a, values= ['cta', 'customers'] , index=['category'],
                       columns=['screen_slot_name'])

cta                                      \
screen_slot_name     CaptainSearchScreen HomeScreen arrived onTheWay started   
category                                                                       
dating                              0.99       0.09    0.09     0.20    0.28   
delivery_food                       1.04       0.10    0.09     0.24    0.35   
delivery_grocery                    1.12       0.10    0.09     0.24    0.32   
devotional                          1.11       0.09    0.07     0.20    0.31   
driver_app                          1.15       0.18    0.10     0.28    0.45   
ecommerce                           1.08       0.12    0.10     0.26    0.38   
educational                         0.93       0.09    0.08     0.22    0.34   
entertainment                       0.94       0.09    0.07     0.21    0.29   
fantasy_sports                      1.09       0.12    0.09     0.25    0.35   
finance_investment                  1.02       0.10    0.08     0.23    0.34   
finance_news                        0.97       0.11    0.07     0.20    0.26   
finance_transactions                1.06       0.12    0.09     0.26    0.38   
gaming                              1.05       0.12    0.10     0.27    0.40   
health                              1.06       0.09    0.08     0.21    0.28   
home_security                       0.32       0.00    0.00     0.00    0.00   
home_service                        1.14       0.10    0.08     0.22    0.29   
insurance                           1.09       0.10    0.09     0.26    0.30   
news                                1.11       0.12    0.10     0.26    0.38   
news_finance                        0.96       0.12    0.08     0.20    0.24   
office                              1.03       0.11    0.08     0.23    0.34   
ott                                 1.06       0.12    0.09     0.25    0.38   
social                              1.07       0.13    0.10     0.26    0.39   
streaming_music                     1.03       0.11    0.09     0.25    0.36   
telecom                             1.03       0.19    0.06     0.22    0.37   
tools                               1.07       0.13    0.10     0.26    0.39   
travel_bookings                     1.02       0.11    0.09     0.23    0.35   
travel_metro                        0.88       0.08    0.08     0.24    0.31   
travel_ridehailing                  1.08       0.11    0.09     0.25    0.36   
wearable                            1.04       0.10    0.07     0.20    0.28   

                               customers                                   \
screen_slot_name     CaptainSearchScreen HomeScreen    arrived   onTheWay   
category                                                                    
dating                           84613.0    69411.0    57454.0    96898.0   
delivery_food                  1452183.0  1153403.0   973021.0  1589578.0   
delivery_grocery                806319.0   666171.0   556186.0   912996.0   
devotional                       29171.0    19549.0    17899.0    28696.0   
driver_app                      201893.0   187126.0   140222.0   238533.0   
ecommerce                      1681976.0  1416906.0  1158621.0  1875097.0   
educational                     420259.0   344138.0   283297.0   459529.0   
entertainment                   620351.0   458423.0   382670.0   636258.0   
fantasy_sports                  152998.0   142893.0   117546.0   192162.0   
finance_investment              827057.0   690209.0   557784.0   918470.0   
finance_news                     51274.0    41973.0    32660.0    54784.0   
finance_transactions           1783143.0  1500876.0  1254391.0  2029152.0   
gaming                          767131.0   672017.0   556510.0   889729.0   
health                          217506.0   156713.0   132977.0   220309.0   
home_security                      172.0      129.0      116.0      175.0   
home_service                    472739.0   345365.0   287835.0   484295.0   
insurance    